# 第91课 · 注意力图解 — Self-Attention 权重热力图（Heatmap），LoRA 低秩结构可视化

**目标**：可视化 Transformer attention 的 token 关注模式，展示 LoRA 参数量节省

🔗 Aurora 连接：`aurora.llm`（KVCache、采样、TF-IDF 检索）· 视觉档案（aurora.whisper 尚未实现 — 自注意力可视化将在 aurora.multimodal 模块中）

← **上一课**　[L90 · 对话式 RAG](L90_agent.ipynb)

> 上节课学习了 **对话式 RAG**：会话记忆、来源归因与 Podcast Q&A 流水线。  
> 本课将探讨 **注意力图解**。

## 神经网络读一句话时，它在"看"哪里？

当你读"the **guitar** is an **instrument**"，你的大脑知道"guitar"和"instrument"是最相关的词对——前者是后者的具体实例。

**Transformer 如何做到这一点？** 用 Self-Attention（自注意力）：每个词对所有其他词分配一个"关注权重"。

```
Q = X @ W_Q   # Query：我在问什么？
K = X @ W_K   # Key：其他词是什么？
A = softmax(Q @ Kᵀ / √d_k)   # 权重矩阵：我对谁的关注度最高？
```

结果是一张 `(seq_len × seq_len)` 的权重矩阵，可以直接画成热力图（Heatmap）——横轴是"被关注的词"，纵轴是"发出关注的词"。

**LoRA 的联系**：LoRA 冻结权重矩阵 W（包括 W_Q, W_K），只训练低秩增量 ΔW=B@A。rank r 越小 → 参数越少，但对注意力权重的可调整范围也越有限。

本节：画出热力图直观理解注意力模式，再实现一个 `find_top_pairs()` 分析函数，从权重矩阵中提取最强 token 关联。

## 热力图怎么读？

**对角线亮**（`A[i,i]` 高）→ 词主要关注自身（常见于浅层）

**非对角线亮**（`A[i,j]` 高，i≠j）→ 词 i 强烈依赖词 j 的语义（语言模型中的语法/语义关联）

**随机初始化的 W_Q, W_K** 不携带语义，热力图呈随机花纹；微调后，语义相关词对的权重会显著升高。

**√d_k 的作用**：不除时，高维点积方差大 → softmax 极度尖锐 → 梯度几乎消失；除以 √d_k 把方差归一化到 1。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Attention 热力图

Self-attention 的输出权重矩阵 `A` 形状为 `(seq_len, seq_len)`，
元素 `A[i, j]` 表示第 `i` 个 token 对第 `j` 个 token 的关注程度。

计算流程：
```
Q = X @ W_Q   # (seq, d_k)
K = X @ W_K   # (seq, d_k)
A = softmax(Q @ K.T / sqrt(d_k))   # (seq, seq)
```

热力图中横轴是 Key（被关注的 token），纵轴是 Query（发出关注的 token）。
对角线权重高说明 token 主要关注自身；非对角线的亮点揭示语义关联。

In [ ]:
def scaled_dot_product_attention(X, W_Q, W_K, d_k):
    """手动计算 self-attention 权重矩阵"""
    Q = X @ W_Q
    K = X @ W_K
    scores = Q @ K.T / (d_k ** 0.5)
    scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
    A = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
    return A

np.random.seed(42)
tokens = ['the', 'guitar', 'is', 'an', 'instrument']
seq_len, d_model, d_k = 5, 16, 8
X = np.random.randn(seq_len, d_model)
W_Q = np.random.randn(d_model, d_k)
W_K = np.random.randn(d_model, d_k)

A = scaled_dot_product_attention(X, W_Q, W_K, d_k)
print('Attention matrix shape:', A.shape)
print('每行之和 (应为1.0):', A.sum(axis=1).round(4))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A, cmap='Blues', vmin=0, vmax=A.max())
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticklabels(tokens, fontsize=11)
ax.set_xlabel('Key (被关注)', fontsize=11)
ax.set_ylabel('Query (发出关注)', fontsize=11)
ax.set_title('Self-Attention 热力图\n"the guitar is an instrument"', fontsize=12)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{A[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if A[i,j] > 0.35 else 'black')
plt.tight_layout()
plt.show()
print(f'guitar→instrument: {A[1,4]:.4f}')
print(f'guitar→guitar:     {A[1,1]:.4f}')

## 2. LoRA 参数量柱状图

LoRA 冻结原始权重 `W`（形状 `d×d`），只训练两个低秩矩阵：
```
A: (r, d)   参数量 = r * d
B: (d, r)   参数量 = d * r
ΔW = B @ A  # 等效维度仍是 (d, d)，但只有 2*d*r 个自由度
```
节省比例 = `1 - 2r/d`。当 d=4096（LLaMA-7B / GPT-3 规模）、r=8 时，节省 > 99.6%。

柱状图直观对比：不同 rank 下可训练参数 vs 原始矩阵参数。

In [ ]:
d = 4096
ranks = [1, 2, 4, 8, 16, 32, 64]
total_params = d * d
lora_params = [2 * r * d for r in ranks]
savings_pct = [(1 - lp / total_params) * 100 for lp in lora_params]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
x = range(len(ranks))
bars = ax.bar(x, lora_params, color='steelblue', alpha=0.85, label='LoRA 可训练参数')
ax.axhline(total_params, color='tomato', linewidth=2, linestyle='--',
           label=f'原始矩阵 ({total_params:,})')
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels([f'r={r}' for r in ranks])
ax.set_ylabel('参数量（log 坐标）')
ax.set_title(f'LoRA 参数量 vs 原始矩阵\n(d={d})')
ax.legend()
for bar, lp in zip(bars, lora_params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.4,
            f'{lp:,}', ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
ax2.bar(x, savings_pct, color='seagreen', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([f'r={r}' for r in ranks])
ax2.set_ylabel('参数节省比例 (%)')
ax2.set_ylim(95, 100.5)
ax2.set_title('LoRA 节省比例 (d=4096)')
for i, sp in enumerate(savings_pct):
    ax2.text(i, sp + 0.05, f'{sp:.2f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()
print(f'r=8 节省: {savings_pct[3]:.2f}%  可训练: {lora_params[3]:,} / {total_params:,}')

## 参数实验：guitar-instrument 注意力权重（Attention Weights）

**句子**：`"the guitar is an instrument"`

**实验参数**：`exp_d_k`（注意力键维度）、`exp_seed`（随机种子）

**预期现象**：随机初始化的 W_Q / W_K 不携带语义，`guitar→instrument` 的权重
不一定高于其他 token 对。改变 `exp_d_k` 时：
- `d_k` 越小 → softmax 输入方差越大 → 分布越尖锐（某个位置权重接近1）
- `d_k` 越大 → 输入除以更大的 `sqrt(d_k)` → 分布越均匀（接近均匀分布）
在语言模型微调后的权重中，语义相关 token 对的注意力权重会显著高于无关对。

In [ ]:
# 修改 exp_d_k 或 exp_seed，观察 guitar→instrument 权重变化
exp_d_k = 4       # 试试 4 / 8 / 32 / 64
exp_seed = 0      # 改变种子观察随机波动

np.random.seed(exp_seed)
X_exp = np.random.randn(seq_len, d_model)
W_Q_exp = np.random.randn(d_model, exp_d_k)
W_K_exp = np.random.randn(d_model, exp_d_k)
A_exp = scaled_dot_product_attention(X_exp, W_Q_exp, W_K_exp, exp_d_k)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A_exp, cmap='Blues', vmin=0, vmax=A_exp.max())
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticklabels(tokens, fontsize=11)
ax.set_title(f'Attention 热力图 d_k={exp_d_k}, seed={exp_seed}', fontsize=12)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{A_exp[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if A_exp[i,j] > 0.35 else 'black')
plt.tight_layout()
plt.show()

guitar_idx, instrument_idx = 1, 4
print(f'd_k={exp_d_k}: guitar→instrument = {A_exp[guitar_idx, instrument_idx]:.4f}')
print(f'd_k={exp_d_k}: guitar→guitar     = {A_exp[guitar_idx, guitar_idx]:.4f}')
print('提示：d_k 越大 softmax 分布越均匀，越小分布越尖锐')

## ✏️ 练习：`find_top_pairs(A, tokens, k=3)`

**任务**：从 `(seq, seq)` 注意力权重矩阵中，提取关注权重最高的 k 个 (query, key) token 对。

**签名**：
```python
def find_top_pairs(A: np.ndarray, tokens: list, k: int = 3) -> list[tuple]:
    # 返回: [(query_token, key_token, weight), ...] 按 weight 降序
```

**3步实现路线**：

| 步骤 | 操作 | 代码提示 |
|---|---|---|
| 1 | 展开所有 (i,j) 对及其权重 | `np.ndindex(A.shape)` 或双重循环 |
| 2 | 排序 | `sorted(..., key=lambda x: -x[2])` |
| 3 | 取前 k 个，映射 index→token | `tokens[i], tokens[j], float(A[i,j])` |

**验收标准**：
- 返回列表长度恰好为 k
- 第一项权重最高（降序）
- 元素格式为 `(str, str, float)`

In [ ]:
def find_top_pairs(A: np.ndarray, tokens: list, k: int = 3) -> list:
    """从注意力权重矩阵提取最强 token 关联对。

    Returns
    -------
    list of (query_token: str, key_token: str, weight: float)
        按 weight 降序，长度为 k。
    """
    # TODO: 遍历所有 (i,j) 对，按 A[i,j] 降序排序，取前 k 个
    raise NotImplementedError("请实现 find_top_pairs")


# ── 验证 ──────────────────────────────────────────────────────────────────────
try:
    top_pairs = find_top_pairs(A, tokens, k=3)
    assert len(top_pairs) == 3, f"应返回3对，得到{len(top_pairs)}"
    assert all(len(p) == 3 for p in top_pairs), "每项应为(query, key, weight)三元组"
    q, k_tok, w = top_pairs[0]
    assert isinstance(q, str) and isinstance(k_tok, str), "前两项应为字符串"
    assert isinstance(w, float), "第三项应为浮点数权重"
    assert w >= top_pairs[1][2] >= top_pairs[2][2], "应按权重降序排列"
    print(f"✅ find_top_pairs 验证通过，最强关联：")
    for q_tok, k_tok, weight in top_pairs:
        print(f"   {q_tok:10s} → {k_tok:10s}  权重 {weight:.4f}")
except NotImplementedError:
    print("⚠️ 尚未实现 find_top_pairs")

## ✏️ 闭卷推导检查格 — 注意力数学

**规则：关闭上方所有格，仅凭记忆推导。**

**题目**：序列长度 seq=3，d_k=4，输入矩阵 X 形状 (3, 16)。

1. Q = X @ W_Q 的形状是？（W_Q: 16×4）
2. scores = Q @ Kᵀ 的形状是？
3. 除以 √d_k 后，若 scores[0]=[2.0, 1.0, 0.5]，写出 softmax 计算步骤（含数值稳定处理）
4. A.sum(axis=-1) 的值应该是多少？（每行）
5. LoRA 参数节省公式：d=512, r=8 时，原始矩阵参数量 vs LoRA 参数量 vs 节省比例？

（在此处写推导...）

In [ ]:
# 验证：注意力数学手算
import numpy as np

# 1-2. shape verification
_seq, _d_model, _d_k = 3, 16, 4
_X = np.random.randn(_seq, _d_model)
_W_Q = np.random.randn(_d_model, _d_k)
_W_K = np.random.randn(_d_model, _d_k)
_Q = _X @ _W_Q
_K = _X @ _W_K
assert _Q.shape == (_seq, _d_k), f"Q shape应({_seq},{_d_k})，得{_Q.shape}"
print(f"1 ✅  Q = X @ W_Q: ({_seq},{_d_model}) @ ({_d_model},{_d_k}) = ({_seq},{_d_k})")

_scores = _Q @ _K.T
assert _scores.shape == (_seq, _seq), f"scores shape应({_seq},{_seq})，得{_scores.shape}"
print(f"2 ✅  scores = Q @ Kᵀ: ({_seq},{_d_k}) @ ({_d_k},{_seq}) = ({_seq},{_seq})")

# 3. softmax calculation
_raw = np.array([2.0, 1.0, 0.5])
_stable = _raw - _raw.max()
_exp = np.exp(_stable)
_probs = _exp / _exp.sum()
assert abs(_probs.sum() - 1.0) < 1e-9
print(f"3 ✅  softmax([2,1,0.5]): stable={_stable}, exp={_exp.round(4)}, probs={_probs.round(4)}")

# 4. row sums
_A_test = scaled_dot_product_attention(_X, _W_Q, _W_K, _d_k)
assert np.allclose(_A_test.sum(axis=-1), 1.0, atol=1e-9), "每行和应为1"
print(f"4 ✅  A.sum(axis=-1) 全为 1.0（softmax 归一化保证）")

# 5. LoRA savings
d_lora, r_lora = 512, 8
orig = d_lora * d_lora
lora_p = 2 * d_lora * r_lora
saving = (1 - lora_p / orig) * 100
assert abs(saving - (1 - 2*r_lora/d_lora)*100) < 1e-9
print(f"5 ✅  d={d_lora},r={r_lora}: 原始={orig:,}, LoRA={lora_p:,}, 节省={saving:.2f}%")

## 本课收束

`scaled_dot_product_attention` 输出的 `(seq, seq)` 权重矩阵，直接可视化为热力图，揭示每层 token 之间的注意力分配模式。
LoRA 的参数节省来自 `ΔW = B @ A` 的低秩分解（Low-Rank Decomposition），rank r 决定可训练参数量 `2·d·r`，这两个核心量都将进入 `aurora.llm` 的微调模块。
本模块 LLM 应用告一段落。

In [ ]:
# ✏️ 本课自评
l91_review = {
    "attention_heatmap_reading":  None,  # 会读热力图：对角线/非对角线，语义关联模式？True/False
    "sqrt_dk_rationale":          None,  # 理解√d_k：防止高维点积方差大→softmax饱和？True/False
    "find_top_pairs_impl":        None,  # find_top_pairs()实现正确，3对验证通过？True/False
    "lora_savings_formula":       None,  # 能口算LoRA参数节省：2·d·r vs d²，节省=(1-2r/d)×100%？True/False
    "whiteboard_passed":          None,  # 白板挑战注意力数学推导通关？True/False
}

unfilled = [k for k, v in l91_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l91_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L91 全部通关！LLM 模块完成，进入 L92：端到端流水线')

---

→ **下一课**　[L92 · 端到端流水线](../10_integration/L92_pipeline.ipynb)

> 下节课将学习 **端到端流水线**：麦克风 → ASR → LLM → 文本回答，全链路组装。